# Replay scorecards, source truth and every pinned pipeline dependency


This additional check verifies the first rank-based pilot before its commit. Donor/independent biological admission is not implied; the target is stored classifier output.

In [1]:
import json, hashlib
from pathlib import Path
import pandas as pd
from starplast import artifacts as A, record_scorecards as R, baselines as B
from starplast.splits import read_split
out = Path('results/label_knn_pilot_2026_10_08_v5')
split, exclusions = read_split(out/'split.json')
manifest = json.loads((out/'held_out/manifest.json').read_text())
spec = A.spec_from_dict(manifest['spec'])
artifact = A.read_artifact(out/'held_out', expected=spec, split=split)
names={'label_adapter':'label_records.py', 'native_vote':'strategies.py','scorecard_aggregation':'record_scorecards.py', 'standard_metrics':'scorecard.py','baseline_recipes':'baselines.py'}
for dependency in spec.dependencies:
    if dependency.kind=='code':
        path=Path('scripts/freeze_label_knn_pilot.py') if dependency.name=='pilot_assembler' else Path('starplast')/names[dependency.name]
        assert hashlib.sha256(path.read_bytes()).hexdigest()==dependency.sha256
        archived=out/'code'/path.name
        if not archived.exists(): archived.write_bytes(path.read_bytes())
row_columns=['entity','prediction','support','abstained','calibrated_confidence','truth','correct','group']
assert all(set(row)==set(row_columns) for row in artifact.payloads['rows.json'])
rows=pd.DataFrame(artifact.payloads['rows.json'],columns=row_columns)
scores=pd.DataFrame(artifact.payloads['class_scores.json']['scores'],columns=artifact.payloads['class_scores.json']['classes'])
scope_data=dict(artifact.payloads['card.json']['scope'])
scope_data['gaps']=tuple(scope_data['gaps'])
scope=R.RecordScope(**scope_data)
normalize=lambda value:json.loads(json.dumps(value,allow_nan=False))
assert normalize(R.aggregate(rows,scope,parameters={'class_scores':scores}))==artifact.payloads['card.json']
assert normalize(R.class_cards(rows,scope))==artifact.payloads['class_cards.json']
state=artifact.payloads['model_state.json']
training_labels=pd.Series(state['training_labels'],index=state['fit_entities'])
baseline=B.label_baselines(training_labels,split,seed=17)
for column in ('majority','prevalence_call'):
    baseline_rows=rows.drop(columns=['prediction','correct','abstained']).assign(prediction=baseline[column].to_numpy())
    assert normalize(R.aggregate(baseline_rows,scope))==artifact.payloads['baseline_cards.json'][column]
{'cohort':len(rows),'cards':'exact','baselines':'same cohort and training labels','code_dependencies':'all matching'}

{'cohort': 560, 'cards': 'exact', 'baselines': 'same cohort and training labels', 'code_dependencies': 'all matching'}